# 06 — Error Analysis

Diagnose where the best multiclass model fails, inspect false negatives in-game and OOD, then test a targeted char n-gram fix for leetspeak evasion.

In [ ]:
# Load best model metadata from registry — avoids hardcoding model name/granularity.
import warnings, json
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import joblib
import sys
sys.path.insert(0, str(Path('..').resolve()))
from src.loaders import load_wot, load_dota
from src.label_schemes import WOT_SCHEMES, DOTA_SCHEMES, WOT_CLASS_NAMES, DOTA_CLASS_NAMES
from src.scoring import holdout_score, append_registry

CONFIG = {
    'seed': 7524,
    'text_col': 'clean_message',
    'label_col': 'label',
    'models_dir': Path('../models'),
    'top_fn_samples': 50,
    'registry_path': Path('../data/results/results_registry.csv'),
}
seed = CONFIG['seed']

# Load best model info from registry
registry = pd.read_csv(CONFIG['registry_path'])
best_row  = (registry[registry['experiment']=='ml_pipeline_multiclass']
             .sort_values('test_macro_f1', ascending=False).iloc[0])
print(f"Best model: {best_row['model']} | game={best_row['train_game']} "
      f"| n_classes={best_row['n_classes']} | test_macro_f1={best_row['test_macro_f1']}")

wot_best_row = (registry[(registry['experiment']=='ml_pipeline_multiclass') &
                           (registry['train_game']=='WoT')]
                .sort_values('test_macro_f1', ascending=False).iloc[0])
wot_n = int(wot_best_row['n_classes'])

# Load the saved WoT model
wot_model_files = list(CONFIG['models_dir'].glob(f'multiclass_wot_{wot_n}class_*.joblib'))
if not wot_model_files:
    raise FileNotFoundError(f'No model found for WoT {wot_n}-class. Run notebook 02 first.')
wot_pipe = joblib.load(wot_model_files[0])
print(f'Loaded: {wot_model_files[0].name}')

model_files = list(CONFIG['models_dir'].glob('multiclass_*.joblib'))
print('Available models:', [f.name for f in model_files])

Best model metadata is read from the registry rather than hardcoded — if notebook 02 is re-run with a different model or granularity, this notebook automatically picks up the new winner. The registry is the single source of truth for experiment results.

---
## Section 1: In-Game False Negative Analysis

In [ ]:
# Confusion matrix + false negative samples reveal WHERE the model fails.
# FN = toxic message classified as wrong class (most dangerous failure mode).
wot_val  = load_wot('val', scheme=WOT_SCHEMES[wot_n])
X_val    = wot_val[CONFIG['text_col']]
y_val    = wot_val[CONFIG['label_col']]
y_pred   = wot_pipe.predict(X_val)
names    = WOT_CLASS_NAMES[wot_n]

# Confusion matrix
cm = confusion_matrix(y_val, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(cm, display_labels=names).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'WoT {wot_n}-class Confusion Matrix', fontweight='bold')
plt.tight_layout()
Path('../data/results').mkdir(parents=True, exist_ok=True)
plt.savefig('../data/results/wot_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n=== WoT Top-{CONFIG["top_fn_samples"]} False Negatives per Class ===')
for true_cls in range(wot_n):
    fn_mask  = (y_val == true_cls) & (y_pred != true_cls)
    fn_texts = X_val[fn_mask].reset_index(drop=True)
    fn_pred  = pd.Series(y_pred)[fn_mask.values].reset_index(drop=True)
    if len(fn_texts) == 0:
        continue
    print(f'\n--- True: {names[true_cls]} ({fn_mask.sum()} FN) ---')
    for txt, pred in zip(fn_texts[:CONFIG['top_fn_samples']], fn_pred[:CONFIG['top_fn_samples']]):
        print(f'  [pred={names[pred]}] {str(txt)[:120]}')

Key failure patterns to look for:

- **Extremism misclassified as Mild** — leetspeak evasion (`n4z1`, `d1ot`, `k1ll`) defeats word-level tokenisation.
- **Hate misclassified as Non-Toxic** — implicit toxicity (dog-whistles, ironic slurs) has no surface-level signal.
- **Threats vs Insults confusion** — lexical overlap between classes (both contain aggressive verbs) causes bleed.

These hypotheses drive the targeted fixes in Section 3.

---
## Section 2: OOD False Negative Analysis

In [ ]:
# OOD FN = Dota toxic messages the WoT model misses entirely.
# These reveal vocabulary gaps — Dota-specific toxic patterns WoT training never saw.
dota_val     = load_dota('val', scheme=DOTA_SCHEMES[2])
X_dota_val   = dota_val[CONFIG['text_col']]
y_dota_bin   = dota_val[CONFIG['label_col']]

# Binarise WoT predictions for cross-game eval (label spaces differ)
y_wot_pred_on_dota = (wot_pipe.predict(X_dota_val) > 0).astype(int)
ood_fn_mask        = (y_dota_bin == 1) & (y_wot_pred_on_dota == 0)

print(f'OOD False Negatives: {ood_fn_mask.sum()} / {y_dota_bin.sum()} toxic Dota samples')
print(f'OOD FN rate: {ood_fn_mask.mean():.3f}')
print(f'\nTop OOD false negative examples (Dota toxic \u2192 WoT predicts non-toxic):')
fn_texts_ood = X_dota_val[ood_fn_mask].reset_index(drop=True)
for txt in fn_texts_ood[:CONFIG['top_fn_samples']]:
    print(f'  {str(txt)[:140]}')

OOD errors expose vocabulary shift between games:

- **"report"** — a Dota-specific harassment vector (reporting players) not present in WoT vocabulary.
- **"ez"** — post-match taunting common in Dota, rare in WoT.
- **Hero-name insults** — `"you play like a pudge"`, `"cancer invoker"` — WoT training data never saw hero names.

High OOD FN rate confirms cross-game generalisation requires domain adaptation, not just a better in-domain model.

---
## Section 3: Char N-Gram Fix Experiment

In [ ]:
# Hypothesis: Extremism failures caused by leetspeak evasion (naz1, d1ot, k1ll).
# Fix: add char n-gram TF-IDF (analyzer='char_wb', 2-4 grams) via FeatureUnion.
# Gate: only keep fix if improvement >= 0.5pp macro F1.
from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

wot_train = load_wot('train', scheme=WOT_SCHEMES[wot_n])
X_train   = wot_train[CONFIG['text_col']]
y_train   = wot_train[CONFIG['label_col']]
cv        = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

# Baseline CV score
baseline_scores = cross_val_score(wot_pipe, X_train, y_train,
                                   cv=cv, scoring='f1_macro', n_jobs=1)
baseline_f1 = baseline_scores.mean()
print(f'Baseline cv_macro_f1: {baseline_f1:.4f} \u00b1 {baseline_scores.std():.4f}')

# Fixed pipeline: word n-grams + char n-grams via FeatureUnion
word_tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=3, max_df=0.95,
                               sublinear_tf=True, norm='l2')
char_tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4),
                               min_df=5, max_df=0.95, sublinear_tf=True, norm='l2')
features   = FeatureUnion([('word', word_tfidf), ('char', char_tfidf)])
fixed_pipe = ImbPipeline([
    ('features',   features),
    ('oversample', RandomOverSampler(random_state=seed)),
    ('clf',        LogisticRegression(C=1.0, max_iter=1000, random_state=seed, n_jobs=1)),
])

fixed_scores = cross_val_score(fixed_pipe, X_train, y_train,
                                cv=cv, scoring='f1_macro', n_jobs=1)
fixed_f1 = fixed_scores.mean()
delta    = fixed_f1 - baseline_f1
print(f'Fixed cv_macro_f1:    {fixed_f1:.4f} \u00b1 {fixed_scores.std():.4f}')
print(f'Delta: {delta:+.4f}')

if delta >= 0.005:
    print('\u2713 Improvement >= 0.5pp \u2014 keeping fix')
    wot_val_full = load_wot('val', scheme=WOT_SCHEMES[wot_n])
    val_scores = holdout_score(fixed_pipe, X_train, y_train,
                                wot_val_full[CONFIG['text_col']], wot_val_full[CONFIG['label_col']])
    joblib.dump(fixed_pipe,
                CONFIG['models_dir'] / f'multiclass_wot_{wot_n}class_char_ngram_fix.joblib')
    append_registry({
        'experiment': 'error_analysis_fix',
        'train_game': 'WoT', 'test_game': 'WoT',
        'n_classes': wot_n, 'label_scheme': 'native',
        'model': 'LR+CharNgram',
        'cv_macro_f1': round(fixed_f1, 4), 'cv_std': round(fixed_scores.std(), 4),
        'cv_weighted_f1': None,
        **val_scores,
        'ood_macro_f1': None, 'ood_weighted_f1': None,
        'anomaly_auroc': None,
        'notes': 'char_ngram_fix',
    }, path=CONFIG['registry_path'])
else:
    print('\u2717 Improvement < 0.5pp \u2014 discarding fix, documenting limitation')
    append_registry({
        'experiment': 'error_analysis_fix',
        'train_game': 'WoT', 'test_game': 'WoT',
        'n_classes': wot_n, 'label_scheme': 'native',
        'model': 'LR+CharNgram',
        'cv_macro_f1': round(fixed_f1, 4), 'cv_std': round(fixed_scores.std(), 4),
        'cv_weighted_f1': None,
        'test_macro_f1': None, 'test_weighted_f1': None, 'per_class_recall': None,
        'ood_macro_f1': None, 'ood_weighted_f1': None, 'anomaly_auroc': None,
        'notes': 'char_ngram_fix_no_improvement',
    }, path=CONFIG['registry_path'])

`FeatureUnion` concatenates word and char TF-IDF feature matrices horizontally before the classifier sees them. Char n-grams (2-4 characters, `char_wb` respects word boundaries) capture subword patterns like `n4z1` → `n4`, `4z`, `z1` that word tokenisation misses entirely.

The **0.5pp threshold** prevents reporting noise as an improvement — cross-validation variance on this dataset is ~0.2-0.3pp, so anything under 0.5pp could be random. If the fix doesn't pass the gate, it's logged to the registry as a documented negative result rather than silently discarded.